# K. I/O Check

- **K1**: Model save (weights + architecture)
- **K2**: Model load with same performance
- **K3**: Hyperparameters via argparse
- **K4**: Seed reproducibility

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import pickle
import os
import model as model_module

## K1 & K2: Save / Load / Verify

In [ ]:
np.random.seed(42)

mlp = model_module.Model(
    hidden_layer_sizes=[24, 24, 24],
    learning_rate=0.001, epochs=50, batch_size=8,
    solver='adam', output_activation='softmax',
    loss='cross_entropy', weights_initializer='heUniform'
)

x = np.random.randn(100, 30)
y = np.zeros((100, 2))
y[np.arange(100), np.random.randint(0, 2, 100)] = 1
x_tr, x_val = x[:80], x[80:]
y_tr, y_val = y[:80], y[80:]

mlp.fit(x_tr, y_tr, x_val, y_val)

# Predict before save
x_val_norm = (x_val - mlp.mean_train) / mlp.std_train
pred_before = mlp.predict(x_val_norm)

# Save
mlp.save('_test_model.pkl')

# Load
with open('_test_model.pkl', 'rb') as f:
    loaded = pickle.load(f)

x_val_norm2 = (x_val - loaded.mean_train) / loaded.std_train
pred_after = loaded.predict(x_val_norm2)

match = np.allclose(pred_before, pred_after)
print(f'\nPredictions match after load: {match}')
print(f'Status: {"PASS" if match else "FAIL"}')

os.remove('_test_model.pkl')

## K3: Argparse Hyperparameters

In [ ]:
with open('../main.py', 'r') as f:
    src = f.read()

args = ['--data', '--layers', '--epochs', '--batch_size', '--learning_rate', '--split', '--solver', '--seed']
print('Argparse check in main.py:')
for arg in args:
    found = arg in src
    print(f'  {arg:>20}: {"FOUND" if found else "MISSING"} {"PASS" if found else "FAIL"}')

## K4: Full Seed Reproducibility

In [ ]:
def full_run(seed):
    np.random.seed(seed)
    mlp = model_module.Model(
        hidden_layer_sizes=[24, 24, 24],
        learning_rate=0.001, epochs=20, batch_size=8,
        solver='adam', output_activation='softmax',
        loss='cross_entropy', weights_initializer='heUniform'
    )
    x = np.random.randn(50, 30)
    y = np.zeros((50, 2))
    y[np.arange(50), np.random.randint(0, 2, 50)] = 1
    h = mlp.fit(x, y)
    return h['loss'][-1], h['accuracy'][-1]

l1, a1 = full_run(42)
l2, a2 = full_run(42)

print(f'Run 1: loss={l1:.10f} acc={a1:.6f}')
print(f'Run 2: loss={l2:.10f} acc={a2:.6f}')
print(f'Loss match: {l1 == l2} | Acc match: {a1 == a2}')
print(f'Status: {"PASS" if l1 == l2 and a1 == a2 else "FAIL"}')